# qvarnet — a guided tour of the whole package

**qvarnet** is a JAX/Flax toolkit for **Variational Monte Carlo (VMC)** with neural-network
and analytic wavefunctions (NQS), for both **continuous-space** (bosons, fermions, Calogero-
Sutherland, trapped gases) and **discrete/spin** (TFIM) systems.

This notebook is a map of **everything the package offers** — every subsystem, with runnable
core examples and copy-paste patterns for the specialised parts. For a single, fully-annotated
end-to-end run, see the companion **`tutorial.ipynb`**.

### The mental model
```
            MODEL                HAMILTONIAN              SAMPLER
   log|ψ_θ(x)|  (ansatz)   Ĥ = T + V  (energy)    R ~ |ψ|²  (Metropolis)
        │                       │                       │
        └───────────────► train(...) ◄──────────────────┘   + optimizer + configs
                               │
                          TrainResult  ──►  history (per-epoch metrics)
                                            best/final params
                                            diagnostics + observables
```

Everything is **log-space**: a *model* maps `x` of shape `(..., DoF)` to **`log|ψ|`** of shape
`(..., 1)`. VMC minimises `⟨E_loc⟩` where `E_loc = Ĥψ/ψ`.

### Table of contents
1. Discoverability — the registries  ·  2. Models  ·  3. Hamiltonians  ·  4. Samplers  ·
5. A training run  ·  6. Extracting results  ·  7. Observables  ·  8. Diagnostics  ·
9. Advanced training (SR/QGT, TDVP, aux losses, callbacks, masking)  ·  10. Discrete/spin  ·
11. Coordinate systems  ·  12. Persistence  ·  13. Scaling  ·  14. CLI/YAML  ·  15. Reference map

## 1. Discoverability — the registries

Models and Hamiltonians are **registered by name**, so you can list everything available and
build by string (handy for config-driven runs). The two registries plus their lookup helpers:

- Hamiltonians: `qvarnet.list_hamiltonians()`, `qvarnet.define_hamiltonian(name)`, `HAMILTONIAN_REGISTER`
- Models: `qvarnet.models.MODEL_REGISTRY`, `qvarnet.models.get_model(name)`

In [ ]:
from qvarnet.hamiltonian import list_hamiltonians
from qvarnet.hamiltonian.hamiltonian_registry import HAMILTONIAN_REGISTER
from qvarnet.models.registry import MODEL_REGISTRY

print('Registered Hamiltonians:')
for name in sorted(HAMILTONIAN_REGISTER): print('   ', name)
print('\nRegistered Models:')
for name in sorted(MODEL_REGISTRY): print('   ', name)

## 2. Models — the wavefunction ansätze

A model is a Flax `nn.Module` returning `log|ψ|` of shape `(..., 1)`. qvarnet ships several
families, all under `qvarnet.models`:

**Composable (recommended)**
- `LogWavefunction(transform, network, envelope, jastrow, n_particles, n_dim)` — sums
  `network(transform(x)) + envelope(x) + jastrow(x)` in log-space. Mix and match the pieces below.

**Networks** (the `network=` slot)
- `MLP(hidden=[...])` — plain multilayer perceptron on the flat coordinate vector.
- `DeepSet(phi_hidden, F_hidden)` / `DeepSetNoEnvelope` — permutation-invariant `F(mean_i φ(x_i))`.
- `TransformerWavefunction` — permutation-invariant via stacked self-attention.
- `FermionicMLP`, `HalfSpinNonInteractingFermion`, `FermionicMLP2ferms` — Slater-determinant
  (antisymmetric) ansätze for fermions.

**Envelopes** (the `envelope=` slot — confine the wavefunction)
- `GaussianEnvelope()` → `-α²Σxᵢ²`  ·  `PolynomialEnvelope(power=p)` → `-α²Σxᵢᵖ`. `α` is learnable.

**Jastrow** (the `jastrow=` slot — two-body correlations)
- `LogJastrow(n_particles, lambda_init, L=None)` — `λ Σᵢ<ⱼ log|xᵢ-xⱼ|` (open) or the periodic
  Sutherland `log|sin(π Δ/L)|` form when `L` is set.

**Input transforms / layers** (`qvarnet.models.layers`, or `qvarnet.SubtractCM` / `AppendPairwiseDiffs`)
- `SubtractCM` (remove centre of mass), `AppendPairwiseDiffs` (augment with all `xᵢ-xⱼ`), `CustomDense`.

**Analytic / exact** (great for tests & baselines)
- `CalogeroSutherlandAnalyticModel(lambda_init)` — exact CS ground state, one learnable `λ`.
- `LogAnalyticWavefunction`, and the `exponential.py` family (`mlp-gaussian-decay`, etc.).

Every model also exists by registry name (section 1) — e.g. `get_model('deep-set')`.

Let's build a few and confirm the `(..., 1)` log-amplitude contract. (`model.init` returns the
parameter pytree; `model.apply(params, x)` evaluates it.)

In [ ]:
import jax, jax.numpy as jnp
from qvarnet.models.compose import LogWavefunction
from qvarnet.models.mlp import MLP
from qvarnet.models.deep_set import DeepSet
from qvarnet.models.envelopes import GaussianEnvelope
from qvarnet.models.jastrow import LogJastrow
from qvarnet.boundaries import NoBoundary, PeriodicBoundary

N, DIM = 4, 1
x = jnp.ones((5, N * DIM))   # 5 configurations

# (a) MLP + Gaussian envelope + Jastrow, open boundary
m_a = LogWavefunction(transform=NoBoundary(), network=MLP(hidden=[64]),
                      envelope=GaussianEnvelope(),
                      jastrow=LogJastrow(n_particles=N, lambda_init=1.0))

# (b) permutation-invariant DeepSet on a periodic box (note n_particles for the reshape)
m_b = LogWavefunction(transform=PeriodicBoundary(L=5.0), n_particles=N,
                      network=DeepSet(phi_hidden=[32], F_hidden=[32]))

for name, m in [('MLP+env+jastrow', m_a), ('DeepSet/PBC', m_b)]:
    p = m.init(jax.random.PRNGKey(0), x)
    print(f'{name:18s} log|psi| shape = {m.apply(p, x).shape}')

## 3. Hamiltonians — the energy operators

A Hamiltonian supplies `local_energy(params, samples, apply)` = kinetic + potential. You only
construct it; `train()` calls it. Three families:

**Continuous** (`qvarnet.hamiltonian.continuous`)
- `HarmonicOscillatorHamiltonian`, `NN_OscillatorHamiltonian` (nearest-neighbour springs),
  `SoftCoreHamiltonian`, `GrossStructHamiltonian` (electron-nuclear), `CalogeroSutherlandHamiltonian`.

**Continuous with boundary** (`qvarnet.hamiltonian.periodic`, take a `boundary=`)
- `PenetrableSphereHamiltonian` (soft-sphere gas, any `n_dim`), `LatticeBoseHamiltonian` (optical lattice).

**Discrete / spin** (`qvarnet.hamiltonian.discrete`)
- `TFIMHamiltonian` — transverse-field Ising chain.

**Kinetic energy** is `T = -½(Δlog|ψ| + |∇log|ψ||²)` (`hamiltonian/kinetic.py`). The Laplacian Δ is
pluggable via `laplacian_method=`: `'forward_ad'` (default, exact, O(DoF)), `'central_difference'`,
`'full_hessian'` (debug), `'hutchinson'` (stochastic, for large DoF).

**Boundaries** (`qvarnet.NoBoundary`, `qvarnet.PeriodicBoundary(L)`) feed both the model transform
and the boundary-aware Hamiltonians (min-image interactions).

In [ ]:
from qvarnet.hamiltonian.continuous import CalogeroSutherlandHamiltonian, HarmonicOscillatorHamiltonian
from qvarnet.hamiltonian.periodic import PenetrableSphereHamiltonian
from qvarnet.boundaries import PeriodicBoundary

H_cs   = CalogeroSutherlandHamiltonian(L=1.5, epsilon=1e-4)          # 1D inverse-square
H_ho   = HarmonicOscillatorHamiltonian()                            # trap only
H_soft = PenetrableSphereHamiltonian(n_dim=3, R=1.0, V0=2.0,
                                     boundary=PeriodicBoundary(L=3.0))  # 3D soft-sphere gas
print(H_cs); print(H_ho); print(H_soft)

# Swap the Laplacian estimator (e.g. cheaper stochastic trace for very large DoF):
H_cs_hutch = CalogeroSutherlandHamiltonian(L=1.5, epsilon=1e-4,
                                           laplacian_method='hutchinson', hutchinson_n_terms=8)
print('laplacian_method =', H_cs_hutch.laplacian_method)

## 4. Samplers — drawing R ~ |ψ|²

`train()` samples internally, but the samplers are also usable standalone (e.g. to measure
observables from a trained ψ). All in `qvarnet.samplers`:

- `sample_and_process(key, prob_fn, prob_params, init_positions, step_size, n_chains, dof, n_steps,
  burn_in, thinning, box_L, block_size, ...)` — the workhorse: vectorised Metropolis-Hastings over
  `n_chains`, returns `(samples, ...)` of shape `(n_kept, dof)`.
- `sample_parallel_tempering(...)` / `pt_chain` / `geometric_betas` — replica-exchange for rugged
  landscapes (clustered/supersolid regimes). In `train()` enable via `sampler='pt'` in the sampler dict.
- `sample_spins` / `spin_flip_chain` — single-spin-flip MH for discrete systems.
- low-level `mh_chain`, `mh_kernel_log`; diagnostics `autocorr`, `integrated_autocorr_time`,
  `effective_sample_size`, `chain_stats`.

`prob_fn = build_prob_fn(model.apply)` turns `log|ψ|` into the `log|ψ|²` probability the kernel needs.
The `box_L` argument enables the periodic sampler (`0.0` = open boundary). We use this in §7.

## 5. A training run — putting it together

The single entry point is **`train()`**. Its arguments map exactly onto the boxes in the diagram:

```python
train(shape=(n_chains, dof),  # batch geometry
      model, optimizer, hamiltonian,
      training_config,        # TrainingConfig: n_epochs, seed, step-size adaptation, QGT, cusp...
      sampler_params,         # dict OR SamplingConfig: step_size, chain_length, thermalization...
      coord_mode=LabCoords(), # or JacobiCoords()
      select='std', k_best=3, # which/how-many param snapshots to retain
      auxiliary_losses=(),    # e.g. cusp condition
      callbacks=None,         # progress / checkpoint / early-stop hooks
      qgt_config=None, init_params=None)
```

Here is a small, fast, **runnable** CS run (the same physics as `tutorial.ipynb`, fewer epochs).

In [ ]:
import numpy as np, optax
from qvarnet.train import train
from qvarnet.config.training_setup import TrainingConfig
from qvarnet.config.coord_mode import LabCoords
from qvarnet.models.analytic import CalogeroSutherlandAnalyticModel

N, DIM, L = 3, 1, 1.5
DoF = N * DIM
E_exact = N * (1 + L * (N - 1))

model = CalogeroSutherlandAnalyticModel(lambda_init=0.5)   # one learnable param 'lam' -> should reach L
hamiltonian = CalogeroSutherlandHamiltonian(L=L, epsilon=1e-4)

result = train(
    shape=(1024, DoF), model=model, optimizer=optax.adam(3e-3),
    hamiltonian=hamiltonian,
    training_config=TrainingConfig(n_epochs=300, rng_seed=0, warm_walkers=True,
                                   is_update_step_size=True, checkpoint_path='./checkpoints/tour'),
    sampler_params={'step_size': 0.4, 'chain_length': 21, 'thermalization_steps': 20, 'thinning_factor': 1},
    coord_mode=LabCoords(), select='energy', k_best=3,
)
print(result, '  exact E0 =', round(E_exact, 4))

## 6. Extracting results from `TrainResult`

(Full detail in `tutorial.ipynb` §9.) The essentials:

- **History** `result.history` — per-epoch metrics. Iterate (`s.energy`, `s.std`, `s.acceptance_rate`,
  `s.step_size`, `s.step`, `s.error_of_mean`, `s.cm_mean`...), or stack: `result.history.get('energy')`.
- **Parameters** `result.final_params`, `result.best_params()`, `result.best_k_params(n)`,
  `result.best_k(n)` (dicts `{step, metric, params}`), `result.best_steps(n)`.
- **Ranking & verdict** `result.best(n, metric)`, `result.diagnose()` (three-referee convergence report).

In [ ]:
E = result.history.get('energy')
tail = E[-100:]
print(f'VMC E = {tail.mean():.4f} +/- {tail.std()/np.sqrt(len(tail)):.4f}   (exact {E_exact:.4f})')

best = result.best_params()
print('learned lambda  =', float(best['params']['lam']), '  (target', L, ')')
print('best 3 epochs   =', result.best_steps(3))
_ = result.diagnose()   # prints the convergence verdict

## 7. Observables — physics from a trained ψ

`qvarnet.observables` computes structure from samples drawn from the trained wavefunction
(all host/numpy, currently 1-D unless noted):

- `density_histogram` — single-particle density `n(x)` (∫n dx = N).
- `pair_correlation` — pair-distance distribution → `g(r)`.
- `structure_factor`, `commensurate_k` — `S(k)` and the allowed wavevectors of a periodic box.
- `obdm_grid`, `natural_orbitals`, `condensate_fraction`, `obdm_displacement` — one-body density
  matrix `ρ₁(x,x')`, its natural orbitals/occupations, and the condensate fraction `n₀`.
- `blocking_error`, `mean_and_error` — Flyvbjerg-Petersen blocking error for correlated series.

First draw samples from the trained ψ with the §4 sampler, then feed them to the observables.

In [ ]:
from qvarnet.vmc.probability import build_prob_fn
from qvarnet.samplers import sample_and_process
from qvarnet.observables import density_histogram, pair_correlation, structure_factor, commensurate_k
from qvarnet.observables.base import mean_and_error

prob_fn = build_prob_fn(model.apply)
x0 = jax.random.normal(jax.random.PRNGKey(1), (1024, DoF))
samples, _, _ = sample_and_process(key=jax.random.PRNGKey(2), prob_fn=prob_fn, prob_params=best,
                                   init_positions=x0, step_size=0.4, n_chains=1024, dof=DoF,
                                   n_steps=400, burn_in=150, thinning=2, box_L=0.0)
samples = np.asarray(samples)            # (n_kept, DoF)

cx, nx = density_histogram(samples, n_particles=N, bins=50)
cr, gr = pair_correlation(samples, n_particles=N, bins=50)
print('density integral  ~', np.trapezoid(nx, cx), '(should be ~N =', N, ')')
print('<x^2> per particle =', np.mean(samples**2))
print('mean & blocking err of E_loc proxy <x^2>:', mean_and_error((samples**2).mean(axis=1)))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(cx, nx); ax[0].set_title('density n(x)'); ax[0].set_xlabel('x')
ax[1].plot(cr, gr); ax[1].set_title('pair distance dist.'); ax[1].set_xlabel('r')
plt.tight_layout(); plt.savefig('tour_observables.png', dpi=110); plt.show()

## 8. Diagnostics — did it converge / mix?

`qvarnet.diagnostics` is a toolbox of statistical referees (beyond `result.diagnose()`):

- **MCMC mixing**: `iat_geyer` (integrated autocorr time), `ess`, `split_rhat` (Gelman-Rubin),
  `autocorr`.
- **Stationarity**: `geweke_z`, `heidelberger_welch_t`, `is_stationary`; `StationarityStopper`
  (a callback that ends training when stationary).
- **Verdict**: `three_referee_verdict`, `format_verdict`, `v_score` (the dimensionless quality score).
- **Gradients**: `global_grad_norm`, `per_layer_grad_norms`, `gradient_snr`.
- **Parameters**: `global_theta_ratio`, `theta_ratios`, `dead_fraction` (frozen-weight fraction).
- **QGT spectrum**: `qgt_eigenvalues`, `d_eff`, `d_part` (effective dimensionality of the model).
- **Comparison & plots**: `welch_t_test` (compare two runs), `plot_dashboard(result, exact_energy)`.

In [ ]:
from qvarnet.diagnostics import iat_geyer, ess, split_rhat, v_score, is_stationary

E = result.history.get('energy')
print('integrated autocorr time of E(epoch):', round(float(iat_geyer(E)), 2))
print('effective sample size               :', round(float(ess(E)), 1))
print('energy trace stationary?            :', bool(is_stationary(E)))
print('V-score (lower is better)           :', round(float(v_score(result.history, n_particles=N)), 4))
# split_rhat wants multiple chains; here we fake 4 sub-chains from the tail to show the call:
tail = E[-200:].reshape(4, -1)
print('split-Rhat across 4 tail sub-chains :', round(float(split_rhat(tail)), 4))

`plot_dashboard` assembles energy/std/acceptance/step-size panels in one figure:
```python
from qvarnet.diagnostics import plot_dashboard
fig = plot_dashboard(result, exact_energy=E_exact, title='CS run')
fig.savefig('dashboard.png', dpi=120)
```

## 9. Advanced training

All of the following plug into the **same** `train()` call. These are configuration patterns
(not executed here to keep the tour fast).

### 9a. Stochastic Reconfiguration / natural gradient (QGT)
Second-order optimisation via the Quantum Geometric Tensor — often essential for deep models.
```python
from qvarnet.geometry import QGTConfig, DEFAULT_QGT_CONFIG, MEMORY_EFFICIENT_QGT_CONFIG
train(..., training_config=TrainingConfig(n_epochs=..., use_qgt=True),
      qgt_config=QGTConfig(learning_rate=1e-2, regularization=1e-3, solver='cholesky'))
# When use_qgt=True the passed optax optimizer is ignored; SR uses SGD(η=qgt learning_rate).
# Solvers: 'direct' | 'cholesky' | 'gmres' | 'diagonal'; minSR variant for n_params >> n_samples.
```

### 9b. Imaginary-time evolution (TDVP) — `qvarnet.geometry.tdvp`
`tdvp_force`, `imaginary_time_step`, `tdvp_residual` drive the state toward the ground state by
McLachlan's variational principle (an alternative to energy-gradient descent).

### 9c. Auxiliary losses — `qvarnet.losses`
Add physics-informed penalties on top of the VMC energy. The shipped one is the **cusp condition**:
```python
from qvarnet.config.training_setup import CuspConfig
cfg = TrainingConfig(n_epochs=..., cusp=CuspConfig(alpha=0.01, n=2, C_n=L, n_configs_per_pair=10))
# enforces ∂log|ψ|/∂r_ij → C_n/ε^(n/2) as particles coalesce. (Generic API: AuxiliaryLoss subclass
# passed via train(..., auxiliary_losses=(MyLoss(),)).)
```

### 9d. Callbacks — `qvarnet.callbacks`
Training-loop hooks: `ProgressCallback` (tqdm bar), `CheckpointCallback`, `NaNCallback`,
`RunOutputCallback`, `EarlyStopCallback`, `SnapshotCallback` (parameter retention policy),
`StationarityStopper`.
```python
from qvarnet.callbacks import ProgressCallback, EarlyStopCallback, CheckpointCallback
train(..., callbacks=[ProgressCallback(update_every=10), CheckpointCallback(save_every=100),
                      EarlyStopCallback(patience=200)])
```

### 9e. Sparse / masked updates — `qvarnet.vmc.masking`
`mask_updates(tx, mask)`, `magnitude_mask(params, sparsity)`, `random_update_masking(tx, drop_prob)`
wrap an optimizer to freeze or randomly drop a fraction of parameter updates (lottery-ticket style).

### 9f. Robust statistics over seeds — `qvarnet.vmc.multi_seed`
`multi_seed_run(seeds, ...)` trains once per seed and reports R̂ across the tail energy windows.

### 9g. Parallel tempering sampler
For barrier-crossing (clustered states), set in the sampler dict:
`{'sampler': 'pt', 'pt_n_replicas': 6, 'pt_beta_min': 0.05, 'swap_every': 1, ...}`.

## 10. Discrete / spin systems

Beyond continuous space, qvarnet does lattice spin models:

- `TFIMHamiltonian` (transverse-field Ising) — spins `s ∈ {±1}^N` fed as a flat vector.
- `qvarnet.vmc.discrete_train.train_discrete(model, hamiltonian, optimizer, ...)` — single-spin-flip VMC.
- `qvarnet.vmc.full_sum` — **exact** small-system training/energy by summing all `2^N` configs
  (`full_sum_energy`, `train_full_sum`) — no Monte Carlo noise; great for validation.
- `qvarnet.utils.exact_diag` — dense exact diagonalisation references (`tfim_dense`,
  `tfim_ground_state_energy`, `ground_state_energy`).

Below: an exact-diagonalisation benchmark (cheap, no training) you can validate an NQS against.

In [ ]:
from qvarnet.utils.exact_diag import tfim_ground_state_energy
for n in (4, 6, 8):
    E0 = tfim_ground_state_energy(n, J=1.0, h=1.0, pbc=True)
    print(f'TFIM N={n}: exact ground-state energy = {E0:.6f}')

## 11. Coordinate systems — `qvarnet.config.coord_mode`

`coord_mode=` controls the frame the **sampler** works in while the **model** still sees lab coords:
- `LabCoords()` (default) — sampler and model both in Cartesian lab coordinates.
- `JacobiCoords()` — sampler walks in `N` Jacobi *relative* coordinates (centre-of-mass removed),
  the model receives the reconstructed `N+1` lab coords. Removes the trivial CM drift for trapped /
  translationally-invariant systems. (Transforms live in `qvarnet.config.jacobi`.)

```python
from qvarnet.config.coord_mode import JacobiCoords
train(..., coord_mode=JacobiCoords())
```
The model-side helpers `SubtractCM` / `AppendPairwiseDiffs` (§2) address the same CM/relative-
coordinate concerns inside the network.

## 12. Persistence & reproducibility — `qvarnet.utils.checkpoint`

- `TrainingConfig(checkpoint_path=..., save_checkpoints=True)` writes a rolling checkpoint; a run
  **auto-resumes** if a checkpoint already exists at that path.
- `save_run_config(...)` writes a `run_config.json` capturing model name/args, sample shape,
  coord mode and training config — everything needed to reconstruct the run (pass `model_name` and
  `model_args` to `train()` to trigger it).
- `load_run(path)` rebuilds a trained run from disk (params + config) — no manual archaeology.
- `save_checkpoint` / `load_checkpoint` operate on the Flax `VMCState` directly.

```python
from qvarnet.utils import load_run
state, cfg = load_run('./checkpoints/tour')   # reconstruct params + how it was trained
```

## 13. Scaling to many devices / large systems — `qvarnet.utils.sharding`

- `make_chain_mesh(n_devices)`, `shard_over_chains(x, mesh)`, `replicate(tree, mesh)`,
  `sharded_mean(local_fn, params, batch, mesh)` — distribute the **chains** axis across GPUs/TPUs
  while replicating parameters (data-parallel VMC).
- `SamplingConfig(block_size=k)` caps peak random-number memory by sampling the chain in blocks
  (must divide `chain_length`).
- For large `DoF`, use `laplacian_method='hutchinson'` (§3) to avoid the O(DoF) exact Laplacian cost.

## 14. Config-driven runs / CLI — `qvarnet.runner`

For reproducible experiments without writing a notebook, drive everything from a YAML file:
```bash
qvarnet path/to/config.yaml --set training.n_epochs=5000 --set model.args.hidden=[128,128]
```
`qvarnet.runner.run(config_path, overrides)` builds the model and Hamiltonian **by registry name**
(§1), runs `train()`, and saves outputs. `--set key=value` overrides any config field. This is the
intended path for sweeps / cluster jobs (see the `soft_sphere_gas/` sweep scripts for examples).

## 15. Reference — where everything lives

| Subpackage | What's in it |
|---|---|
| `qvarnet.models` | ansätze: `LogWavefunction`, `MLP`, `DeepSet`, `Transformer*`, `Fermionic*`, `*Envelope`, `LogJastrow`, analytic/exponential; `MODEL_REGISTRY`, `get_model` |
| `qvarnet.hamiltonian` | `*Hamiltonian` classes, `kinetic`, `laplacian` methods, registry (`list_hamiltonians`, `define_hamiltonian`) |
| `qvarnet.boundaries` | `NoBoundary`, `PeriodicBoundary`, boundary-aware bases |
| `qvarnet.samplers` | `sample_and_process`, parallel tempering, discrete spin flips, MCMC diagnostics |
| `qvarnet.vmc` | `train`, `TrainResult`, `VMCState`, `build_prob_fn`, training step, `full_sum`, `discrete_train`, `multi_seed`, `masking`, `metrics_history` |
| `qvarnet.config` | `TrainingConfig`, `SamplingConfig`, `CuspConfig`, `LabCoords`/`JacobiCoords`, jacobi transforms |
| `qvarnet.geometry` | QGT (`compute_qgt`, natural gradient, `QGTConfig`), TDVP |
| `qvarnet.losses` | `AuxiliaryLoss`, `CuspLoss`, cusp config generators |
| `qvarnet.callbacks` | `Progress`/`Checkpoint`/`NaN`/`RunOutput`/`EarlyStop`/`Snapshot` callbacks |
| `qvarnet.diagnostics` | MCMC/stationarity/verdict/gradient/parameter/QGT-spectrum diagnostics, `plot_dashboard` |
| `qvarnet.observables` | `density_histogram`, `pair_correlation`, `structure_factor`, OBDM & condensate fraction, blocking error |
| `qvarnet.utils` | checkpointing & `load_run`, `exact_diag`, custom-module loading, numerical gradients, sharding |
| `qvarnet.runner` | YAML/CLI experiment runner |

### Where to go next
- **`tutorial.ipynb`** — one CS run with *every* API call annotated and all result-extraction paths.
- **`../calogero-sutherland/`** — a real CS study (coupling scans, cusp condition).
- **`../soft_sphere_gas/`** — a full production workflow: sweeps, workers, a results DB, `g(r)` analysis.